In [1]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/FRAMES'

import os
os.makedirs(os.path.join(DATA_DIR, 'results'), exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install tqdm -q

In [3]:
import json, os, re, time, collections
from tqdm.auto import tqdm

# Could be changed depending on how many answers we want to evaluate
# Adjust depending on runtime, ideally run all
EVAL_SUBSET = 1


PRED_DIR = os.path.join(DATA_DIR, 'predictions')

def load_predictions(name):
    path = os.path.join(PRED_DIR, f'{name}.json')
    if not os.path.exists(path):
        print(f'WARNING: {path} not found — skipping')
        return None
    with open(path, encoding='utf-8') as f:
        return json.load(f)

# Load all predictions
ALL_MODELS = {
    'B1: BERT no-retrieval':    'b1_bert_no_retrieval',
    'B2: BM25 + BERT':          'b2_bm25_bert',
    'B3: LLM zero-shot':        'b3_llm_zeroshot',
    'B4: LLM few-shot':         'b4_llm_fewshot',
    'C1: Single-hop RAG':       'c1_rag_single_hop',
    'C2: Multi-hop RAG (3hop)': 'c2_rag_multihop_3',
    # 'C3: Multi-hop RAG (5hop)': 'c3_rag_multihop_5',
    # Obviously only uncomment this one if you actually made the C3 model ^
}

predictions = {}
for label, fname in ALL_MODELS.items():
    preds = load_predictions(fname)
    if preds is not None:
        predictions[label] = preds
        print(f'Loaded {label}: {len(preds)} predictions')

if EVAL_SUBSET < 1.0 and predictions:
    import random
    from collections import defaultdict
    random.seed(42)
    first_preds = next(iter(predictions.values()))
    type_indices = defaultdict(list)
    for i, p in enumerate(first_preds):
        key = tuple(sorted(p.get('reasoning_types') or ['Unknown']))
        type_indices[key].append(i)
    subset_indices = set()
    for key, indices in type_indices.items():
        k = max(1, round(len(indices) * EVAL_SUBSET))
        subset_indices.update(random.sample(indices, k))
    predictions = {label: [p for i, p in enumerate(preds) if i in subset_indices]
                   for label, preds in predictions.items()}
    n = len(next(iter(predictions.values())))
    print(f'\nStratified subset: evaluating {n} questions ({EVAL_SUBSET:.0%} of full set)')

Loaded B1: BERT no-retrieval: 412 predictions
Loaded B2: BM25 + BERT: 412 predictions
Loaded B3: LLM zero-shot: 412 predictions
Loaded B4: LLM few-shot: 412 predictions
Loaded C1: Single-hop RAG: 412 predictions
Loaded C2: Multi-hop RAG (3hop): 412 predictions


## Token F1 and Exact Match

In [4]:
def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return text

def token_f1(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    p = num_same / len(pred_tokens)
    r = num_same / len(gold_tokens)
    return 2 * p * r / (p + r)

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def compute_f1_em(preds):
    f1s = [token_f1(p['predicted'], p['gold_answer']) for p in preds]
    ems = [exact_match(p['predicted'], p['gold_answer']) for p in preds]
    return sum(f1s)/len(f1s), sum(ems)/len(ems)

print(f'{'Model':<30}  {'Token F1':>8}  {'EM':>8}')
print('-' * 52)
for label, preds in predictions.items():
    f1, em = compute_f1_em(preds)
    print(f'{label:<30}  {f1:>8.3f}  {em:>8.3f}')

Model                           Token F1        EM
----------------------------------------------------
B1: BERT no-retrieval              0.063     0.000
B2: BM25 + BERT                    0.065     0.032
B3: LLM zero-shot                  0.145     0.044
B4: LLM few-shot                   0.080     0.002
C1: Single-hop RAG                 0.107     0.002
C2: Multi-hop RAG (3hop)           0.120     0.002


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Local Qwen2.5-14B judge
JUDGE_MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'

print(f'Loading judge model {JUDGE_MODEL_ID}...')
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
judge_model.eval()
print('Judge model loaded.')

JUDGE_PROMPT = """You are evaluating whether a model's answer to a factual question is correct.

Question: {question}
Ground truth: {gold}
Model's answer: {predicted}

The model may include reasoning before giving its answer. A response is CORRECT if it contains the correct factual
answer anywhere in the text, even if surrounded by explanation.
A response is INCORRECT if it:
- Never states the correct answer
- States the wrong fact as the answer
- Refuses to answer or says it cannot determine the answer
- Is vague or hedged without committing to a correct answer

Judge an answer based on the Ground truth given only, it does not matter if you agree with the explanation if the prediction doesn't match the ground truth.
Respond with exactly one word: CORRECT or INCORRECT."""

def judge_answer(question, gold, predicted):
    if not str(predicted).strip():
        return False
    prompt = JUDGE_PROMPT.format(question=question, gold=gold, predicted=predicted)
    messages = [{'role': 'user', 'content': prompt}]
    text = judge_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = judge_tokenizer(text, return_tensors='pt').to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=1.0,
            do_sample=False,
        )
    verdict = judge_tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip().upper()
    return verdict == 'CORRECT'

# Checking if the judge works
test_result = judge_answer(
    'What is the capital of France?',
    'Paris',
    'The capital of France is Paris.'
)
print(f'Judge test (should be True): {test_result}')

test_result2 = judge_answer(
    'What is the capital of France?',
    'Paris',
    'What'
)
print(f'Judge test (should be False): {test_result2}')

Loading judge model Qwen/Qwen2.5-14B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Judge model loaded.
Judge test (should be True): True
Judge test (should be False): False


In [6]:
# Run judge on all models.
judge_results = {}


for label, preds in predictions.items():
    safe_label      = label.replace(' ', '_').replace(':', '')
    results_path    = os.path.join(DATA_DIR, 'results', f'judge_{safe_label}.json')
    checkpoint_path = os.path.join(DATA_DIR, 'results', f'judge_{safe_label}_checkpoint.json')

    if os.path.exists(results_path):
        with open(results_path) as f:
            judge_results[label] = json.load(f)
        n_correct = sum(r['judge_correct'] for r in judge_results[label])
        print(f'{label}: loaded from cache ({n_correct}/{len(judge_results[label])} correct)')
        continue

    already_judged = {}
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            for r in json.load(f):
                already_judged[r['question']] = r
        print(f'{label}: resuming checkpoint ({len(already_judged)} already judged)')

    print(f'\nJudging {label} ({len(preds)} predictions)...')
    model_results = []

    for i, p in enumerate(tqdm(preds)):
        if p['question'] in already_judged:
            model_results.append(already_judged[p['question']])
            continue

        correct = judge_answer(p['question'], p['gold_answer'], p['predicted'])
        model_results.append({**p, 'judge_correct': correct})

        if len(model_results) % 25 == 0:
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump(model_results, f, ensure_ascii=False)

    judge_results[label] = model_results

    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(model_results, f, ensure_ascii=False, indent=2)
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    n_correct = sum(r['judge_correct'] for r in model_results)
    print(f'  Accuracy: {n_correct}/{len(model_results)} = {n_correct/len(model_results):.3f}')

B1: BERT no-retrieval: loaded from cache (2/412 correct)
B2: BM25 + BERT: loaded from cache (28/412 correct)
B3: LLM zero-shot: loaded from cache (57/412 correct)
B4: LLM few-shot: loaded from cache (77/412 correct)
C1: Single-hop RAG: loaded from cache (109/412 correct)
C2: Multi-hop RAG (3hop): loaded from cache (121/412 correct)


In [7]:
# Retrieval Recall@k
def title_from_url(url):
    match = re.search(r'/wiki/(.+)', url)
    return match.group(1).replace('_', ' ') if match else url

def retrieval_recall(preds):
    import ast
    recalls = []
    for p in preds:
        if 'gold_wiki_links' not in p or 'retrieved_titles' not in p:
            return None
        links = p['gold_wiki_links']
        if isinstance(links, str):
            try:
                links = ast.literal_eval(links)
            except:
                links = []
        gold = set(title_from_url(u).lower() for u in links)
        retrieved = set(t.lower() for t in (p['retrieved_titles'] or []))
        if gold:
            recalls.append(len(gold & retrieved) / len(gold))
    return sum(recalls) / len(recalls) if recalls else None

In [8]:
# Full results table
print('\n' + '=' * 80)
print('RESULTS TABLE')
print('=' * 80)
print(f'{"Model":<30}  {"LLM-Judge":>9}  {"Token F1":>8}  {"EM":>6}  {"Recall":>7}')
print('-' * 70)

table_rows = []
for label, preds in predictions.items():
    f1, em = compute_f1_em(preds)
    recall = retrieval_recall(preds)
    recall_str = f'{recall:.3f}' if recall is not None else '  N/A'

    if label in judge_results:
        n_correct = sum(r['judge_correct'] for r in judge_results[label])
        judge_acc = n_correct / len(judge_results[label])
        judge_str = f'{judge_acc:.3f}'
    else:
        judge_acc = None
        judge_str = '  N/A'

    print(f'{label:<30}  {judge_str:>9}  {f1:>8.3f}  {em:>6.3f}  {recall_str:>7}')
    table_rows.append({'model': label, 'judge_accuracy': judge_acc, 'token_f1': f1, 'em': em, 'recall': recall})

print('=' * 80)

# Save table
with open(os.path.join(DATA_DIR, 'results', 'main_results.json'), 'w') as f:
    json.dump(table_rows, f, indent=2)
print('Results saved to results/main_results.json')


RESULTS TABLE
Model                           LLM-Judge  Token F1      EM   Recall
----------------------------------------------------------------------
B1: BERT no-retrieval               0.005     0.063   0.000    0.000
B2: BM25 + BERT                     0.068     0.065   0.032    0.437
B3: LLM zero-shot                   0.138     0.145   0.044    0.000
B4: LLM few-shot                    0.187     0.080   0.002    0.000
C1: Single-hop RAG                  0.265     0.107   0.002    0.437
C2: Multi-hop RAG (3hop)            0.294     0.120   0.002    0.616
Results saved to results/main_results.json
